##Project 7: Loss Functions in Neural Networks

## Introduction

- When a neural network makes a prediction, it is rarely perfect — there is always some difference between the actual value and the predicted value
- A **loss function** measures how wrong the prediction is — the higher the loss, the worse the prediction
- During training, the network tries to **minimize this loss** by adjusting its weights through backpropagation
- In this project, the **Iris dataset** is used — the same dataset from Project 6
- Forward propagation is performed to get predicted probabilities, and then two loss functions are calculated: **Mean Squared Error (MSE)** and **Cross-Entropy Loss**

## Task 1: Load Dataset and Get Actual vs Predicted Values

- Load the Iris dataset and prepare actual labels (one-hot encoded)
- Build the same neural network from Project 6
- Run forward propagation to get predicted probabilities
- These actual vs predicted values will be used to calculate loss

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

In [3]:
# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [4]:
# One-hot encode labels
y_encoded = np.eye(3)[y]

In [5]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

In [6]:
# Initialize weights
np.random.seed(42)
W1 = np.random.randn(4, 6)
b1 = np.zeros((1, 6))
W2 = np.random.randn(6, 3)
b2 = np.zeros((1, 3))

In [7]:
# Activation functions
def relu(z):
    return np.maximum(0, z)

def softmax(z):
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [8]:
# Forward propagation
def forward_propagation(X):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)
    return A2

In [10]:
# Get predictions on test set
predicted = forward_propagation(X_test)
actual = y_test

print("Actual labels (first 5 samples):")
print(actual[:5])
print("\nPredicted probabilities (first 5 samples):")
print(np.round(predicted[:5], 4))

Actual labels (first 5 samples):
[[0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 0.]]

Predicted probabilities (first 5 samples):
[[0.3113 0.3804 0.3084]
 [0.5897 0.4094 0.001 ]
 [0.0049 0.9882 0.0069]
 [0.2842 0.4062 0.3096]
 [0.0475 0.9021 0.0504]]


## Task 2: Calculate MSE and Cross-Entropy Loss

### Mean Squared Error (MSE)
- Calculates the **average of squared differences** between actual and predicted values
- Formula: `MSE = mean((actual − predicted)²)`

### Cross-Entropy Loss
- Measures how different the predicted **probability distribution** is from the actual
- Formula: `CE = −sum(actual × log(predicted)) / number of samples`
- A small value (1e-10) is added to avoid log(0) which is mathematically undefined

In [12]:
# Mean Squared Error

def mean_squared_error(actual, predicted):
    return np.mean((actual - predicted) ** 2)

mse_loss = mean_squared_error(actual, predicted)
print("Mean Squared Error (MSE):", round(mse_loss, 4))

Mean Squared Error (MSE): 0.2311


In [13]:
# Cross-Entropy Loss

def cross_entropy_loss(actual, predicted):
    predicted = np.clip(predicted, 1e-10, 1.0)
    return -np.sum(actual * np.log(predicted)) / len(actual)

ce_loss = cross_entropy_loss(actual, predicted)
print("Cross-Entropy Loss:", round(ce_loss, 4))

Cross-Entropy Loss: 1.5872


## Task 3: Compare Results and Explain When to Use Each

| Feature | Mean Squared Error (MSE) | Cross-Entropy Loss |
|---|---|---|
| **Formula** | mean((actual − predicted)²) | −sum(actual × log(predicted)) |
| **Output type** | Any numerical value | Probability (0 to 1) |
| **Best used for** | Regression problems | Classification problems |
| **Penalizes** | Large differences between values | Confident wrong predictions |
| **Example** | House price, marks prediction | Image classification, spam detection |

### When to Use MSE
- Output is a **continuous number** — house price, temperature, student marks
- Output layer uses **Linear activation**

### When to Use Cross-Entropy Loss
- Output is a **class** — Setosa/Versicolor/Virginica, spam/not spam
- Output layer uses **Sigmoid** (binary) or **Softmax** (multi-class)
- Penalizes more when the model is **confidently wrong**
- For this Iris classification problem, **Cross-Entropy is the correct choice**

In [16]:
print("── Loss Function Summary ──\n")
print(f"MSE Loss: {round(mse_loss, 4)}")
print(f"Cross-Entropy Loss: {round(ce_loss, 4)}")

── Loss Function Summary ──

MSE Loss: 0.2311
Cross-Entropy Loss: 1.5872


## Conclusion

- Using the Iris dataset, forward propagation was performed to get real predicted probabilities
- MSE Loss came out to **0.2311** and Cross-Entropy Loss came out to **1.5872**
- The higher Cross-Entropy value does not mean the model is performing badly — Cross-Entropy is naturally higher in scale because it uses logarithms and penalizes wrong probability predictions more strongly
- MSE treats the problem like a number prediction task — it does not fully capture how wrong a classification prediction is
- Cross-Entropy Loss is more meaningful here because it directly measures how far the predicted probability distribution is from the actual class
- Both loss values are high because the weights are randomly initialized and no training has been done — backpropagation and weight updates would bring these values down
- For any classification problem like Iris species prediction, **Cross-Entropy Loss is the correct choice** over MSE